In [ ]:
!pip3 install selenium webdriver-manager pandas openpyxl

In [ ]:
!pip3 install scheduler

In [ ]:
import pandas as pd
import openpyxl
from typing import Optional

def _get_link_if_exists(cell) -> Optional[str]:
    try:
        return cell.hyperlink.target
    except AttributeError:
        return None


def extract_hyperlinks_from_xlsx(file_path, sheet_name=None):
    wb = load_workbook(filename=file_path, data_only=True)

    # 👉 Nếu không truyền sheet_name hoặc sheet không tồn tại → lấy sheet đầu tiên
    if not sheet_name or sheet_name not in wb.sheetnames:
        ws = wb.worksheets[0]
        print(f"⚠️ Sheet '{sheet_name}' không tồn tại → dùng sheet mặc định: '{ws.title}'")
    else:
        ws = wb[sheet_name]

    urls = []
    hotel_names = []
    room_types = []

    for row in ws.iter_rows(min_row=2, max_col=2):
        hotel_cell = row[0]
        room_cell = row[1]

        hotel_names.append(hotel_cell.value)
        urls.append(hotel_cell.hyperlink.target if hotel_cell.hyperlink else "")
        room_types.append(room_cell.value)

    return pd.DataFrame({
        "Hotel": urls,
        "Hotel_name": hotel_names,
        "Room_type": room_types
    })


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
from datetime import datetime, timedelta, date
from openpyxl import load_workbook
import os

# ======= HÀM LẤY LINK TỪ EXCEL ========
def extract_hyperlinks_from_xlsx(file_path, sheet_name=None):
    wb = load_workbook(filename=file_path, data_only=True)
    if sheet_name and sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
    else:
        ws = wb.worksheets[0]
        print(f"ℹ️ Sử dụng sheet mặc định: '{ws.title}'")

    urls, hotel_names, room_types = [], [], []
    for row in ws.iter_rows(min_row=2, max_col=2):
        hotel_value, room_type = row[0], row[1]
        hotel_names.append(hotel_value.value)
        urls.append(hotel_value.hyperlink.target if hotel_value.hyperlink else "")
        room_types.append(room_type.value)
    return pd.DataFrame({"Hotel": urls, "Hotel_name": hotel_names, 'Room_type': room_types})

# ======= TẠO CHROME DRIVER (1 LẦN DUY NHẤT) ========
def create_driver():
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--blink-settings=imagesEnabled=false")
    options.add_experimental_option("prefs", {"profile.managed_default_content_settings.images": 2})
    options.page_load_strategy = "eager"  # Không chờ load hết tài nguyên
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36")
    return webdriver.Chrome(options=options)

# ======= HÀM SET NGÀY QUA CALENDAR UI ========
def set_dates(driver, wait, checkin_date, checkout_date):
    def open_calendar():
        checkin_box = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-selenium='checkInBox']")))
        checkin_box.click()
        time.sleep(0.5)

    def click_next_month_until_visible(target_date):
        for _ in range(12):
            try:
                selector = f"[data-selenium-date='{target_date.strftime('%Y-%m-%d')}']"
                driver.find_element(By.CSS_SELECTOR, selector)
                return
            except:
                try:
                    next_btn = driver.find_element(By.CSS_SELECTOR, "[data-selenium='calendar-next-month-button']")
                    next_btn.click()
                    time.sleep(0.5)
                except Exception as e:
                    print("    ❌ Không tìm được nút next tháng:", e)
                    break

    def select_date(d):
        click_next_month_until_visible(d)
        selector = f"[data-selenium-date='{d.strftime('%Y-%m-%d')}']"
        date_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, selector)))
        driver.execute_script("arguments[0].scrollIntoView(true);", date_button)
        date_button.click()
        time.sleep(0.5)

    try:
        open_calendar()
        select_date(checkin_date)
        select_date(checkout_date)
    except Exception as e:
        print(f"    ❌ Lỗi chọn ngày: {e}")

# ======= HÀM CÀO GIÁ (DÙNG CHUNG DRIVER + CALENDAR UI) ========
def scrape_room_prices(driver, url, checkin_date, checkout_date):
    driver.get(url)
    wait = WebDriverWait(driver, 30)
    results = []

    try:
        # Đóng popup nếu có (chờ tối đa 2s, không phải 30s)
        time.sleep(1)
        try:
            close_btn = WebDriverWait(driver, 2).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, ".ab-close-button"))
            )
            close_btn.click()
        except:
            pass

        # Chọn ngày qua calendar UI
        set_dates(driver, wait, checkin_date, checkout_date)

        # Click search
        search_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-selenium='searchButton']")))
        search_btn.click()

        # Chờ roomGrid load
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div#roomGrid")))
        time.sleep(1)

        # Scroll để trigger lazy load
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)

        room_cards = driver.find_elements(By.CSS_SELECTOR, "div[data-selenium='MasterRoom']")
        for card in room_cards:
            try:
                name = card.find_element(By.CSS_SELECTOR, "[data-selenium='masterroom-title-name']").text
            except:
                name = "NA"
            try:
                price = card.find_element(By.CSS_SELECTOR, "[data-selenium='PriceDisplay']").text
            except:
                price = "NA"
            results.append({"room": name, "price": price})

    except Exception as e:
        page_text = ""
        try:
            page_text = driver.find_element(By.TAG_NAME, "body").text[:200]
        except:
            pass
        if "verify" in page_text.lower() or "human" in page_text.lower():
            print("    ⚠️ CAPTCHA detected")
        else:
            print(f"    ❌ Lỗi: {str(e)[:80]}")

    return results

# ======= MAIN ========
if __name__ == "__main__":
    df_urls = extract_hyperlinks_from_xlsx("./raw.xlsx")

    backup_filename = "hotel_prices_temp.xlsx"
    if os.path.exists(backup_filename):
        print(f"📂 Đang đọc file tạm: {backup_filename}")
        df_prev = pd.read_excel(backup_filename)
    else:
        df_prev = pd.DataFrame(columns=["Hotel", "Room", "Price W1", "Price W2", "Price W3", "Price W4", "Price W5", "Price W6"])

    prev_data = {}
    for _, row in df_prev.iterrows():
        key = (row["Hotel"], row["Room"])
        prev_data[key] = {f"Price W{i}": row.get(f"Price W{i}", "NA") for i in range(1, 7)}

    base_checkin = datetime.today().replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=1)
    base_checkout = base_checkin + timedelta(days=1)
    week_offsets = [0, 7, 14, 21, 28, 35]
    all_week_prices = {}
    max_retries = 3

    # ✅ Mở Chrome 1 lần duy nhất
    driver = create_driver()
    print("🚀 Chrome driver đã sẵn sàng")

    try:
        for index, row in df_urls.iterrows():
            hotel_name = row["Hotel_name"]
            hotel_url = row["Hotel"]
            room_type = row["Room_type"]
            key = (hotel_name, room_type)

            if not hotel_url.startswith("http"):
                print(f"❌ Bỏ qua URL không hợp lệ: {hotel_url}")
                continue

            print(f"\n🏨 {hotel_name} - {room_type}")

            for week_num, offset in enumerate(week_offsets, start=1):
                key_prefix = f"Price W{week_num}"

                if key in prev_data and key_prefix in prev_data[key] and prev_data[key][key_prefix] != "NA":
                    print(f"  ✅ {key_prefix}: đã có, bỏ qua")
                    if key not in all_week_prices:
                        all_week_prices[key] = {}
                    all_week_prices[key][key_prefix] = prev_data[key][key_prefix]
                    continue

                found = False
                for retry in range(max_retries):
                    checkin = base_checkin + timedelta(days=offset + retry)
                    checkout = base_checkout + timedelta(days=offset + retry)
                    checkin_str = checkin.strftime("%Y-%m-%d")
                    checkout_str = checkout.strftime("%Y-%m-%d")

                    print(f"  ⏳ {key_prefix} | {checkin_str} → {checkout_str}" + (f" (retry {retry})" if retry > 0 else ""))

                    data = scrape_room_prices(driver, hotel_url, checkin, checkout)

                    for room_data in data:
                        if room_data["room"].strip() == room_type.strip() and room_data["price"] != "NA":
                            if key not in all_week_prices:
                                all_week_prices[key] = {}
                            all_week_prices[key][key_prefix] = room_data["price"]
                            found = True
                            break

                    if found:
                        break

                if not found:
                    if key not in all_week_prices:
                        all_week_prices[key] = {}
                    all_week_prices[key][key_prefix] = "NA"

            # Lưu tạm sau mỗi khách sạn
            temp_result = {"Hotel": [], "Room": [], **{f"Price W{i}": [] for i in range(1, 7)}}
            for (hotel, room), prices in all_week_prices.items():
                temp_result["Hotel"].append(hotel)
                temp_result["Room"].append(room)
                for i in range(1, 7):
                    temp_result[f"Price W{i}"].append(prices.get(f"Price W{i}", "NA"))
            pd.DataFrame(temp_result).to_excel(backup_filename, index=False)
            print(f"  💾 Đã lưu tạm")

    finally:
        driver.quit()
        print("🛑 Chrome driver đã đóng")

    # Xuất file chính thức
    result = {"Hotel": [], "Room": [], **{f"Price W{i}": [] for i in range(1, 7)}}
    for (hotel, room), prices in all_week_prices.items():
        result["Hotel"].append(hotel)
        result["Room"].append(room)
        for i in range(1, 7):
            result[f"Price W{i}"].append(prices.get(f"Price W{i}", "NA"))

    df = pd.DataFrame(result)
    final_filename = f"hotel_prices_{datetime.today().strftime('%Y%m%d')}.xlsx"
    df.to_excel(final_filename, index=False)
    print(f"\n✅ Saved to: {final_filename}")